# EditPro AI Studio: realistic scene clips on Kaggle's free GPU

Makes one short realistic video clip for every scene in your **EditPro Kaggle pack** with the free, open
**LTX-Video** model (Lightricks, LTX-Video open-weights licence). Your scene picture (if you gave one) becomes the first frame.

**Before you press Run all:**
1. Right side → **Add Input → Upload** → choose `editpro-kaggle-pack.zip` (from EditPro).
2. Right side → **Settings → Accelerator → GPU T4 x2** (or GPU P100), and **Internet → On** (needs a phone-verified Kaggle account).
3. Menu **Run → Run all**. The first time downloads the model (about 10 GB, several minutes). Then each clip takes a few minutes.
4. When it says **Finished**, open the **Output** panel (right side) and download `editpro-clips.zip`.
   In EditPro: Movie Maker → **📥 Add clips from Kaggle**.

If the session stops halfway, just press Run all again: finished clips are kept and skipped.

In [ ]:
# Settings you may change
STEPS = 30          # quality steps (more = better but slower; 20-50)
FPS = 24
MODEL = "Lightricks/LTX-Video"

In [ ]:
# 1) Tools (about 2 minutes)
!pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
# 2) Find the EditPro pack and the graphics card
import glob, json, os, shutil, time, zipfile
import torch

packs = glob.glob("/kaggle/input/**/jobs.json", recursive=True)
if not packs:
    for z in glob.glob("/kaggle/input/**/*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/working/pack")
    packs = glob.glob("/kaggle/working/pack/**/jobs.json", recursive=True)
assert packs, 'EditPro pack not found. Right side: "Add Input" -> Upload -> editpro-kaggle-pack.zip'
PACK = os.path.dirname(packs[0])
jobs = json.load(open(packs[0], encoding="utf-8"))
assert jobs.get("app") == "EditPro" and jobs.get("version") == 1, "This is not an EditPro pack."
print(len(jobs["scenes"]), "scenes to make")
assert torch.cuda.is_available(), "No GPU. Right side: Settings -> Accelerator -> GPU T4 x2 (or P100)"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3) Make the clips (finished ones are skipped, so you can run this again)
from diffusers import LTXImageToVideoPipeline, LTXPipeline
from diffusers.utils import export_to_video, load_image

OUT = "/kaggle/working/clips"
os.makedirs(OUT, exist_ok=True)
NEG = "worst quality, inconsistent motion, blurry, jittery, distorted, deformed face, extra fingers, text, watermark"

def frames_for(seconds):
    n = int(round(seconds * FPS / 8)) * 8 + 1  # LTX needs 8k+1 frames
    return max(25, min(161, n))

pipe, kind_loaded = None, None
order = sorted(jobs["scenes"], key=lambda s: s.get("image") is None)  # picture scenes first: one model switch at most
for s in order:
    out = f"{OUT}/scene-{s['n']}-{s['id']}.mp4"
    if os.path.exists(out):
        print("already made:", os.path.basename(out))
        continue
    kind = "i2v" if s.get("image") else "t2v"
    if kind != kind_loaded:
        pipe = None
        torch.cuda.empty_cache()
        cls = LTXImageToVideoPipeline if kind == "i2v" else LTXPipeline
        print("loading the", "picture-to-video" if kind == "i2v" else "text-to-video", "model…")
        pipe = cls.from_pretrained(MODEL, torch_dtype=torch.bfloat16)
        pipe.enable_model_cpu_offload()
        kind_loaded = kind
    args = dict(prompt=s["prompt"], negative_prompt=NEG, width=s["width"], height=s["height"],
                num_frames=frames_for(s["seconds"]), num_inference_steps=STEPS)
    if kind == "i2v":
        args["image"] = load_image(os.path.join(PACK, s["image"])).convert("RGB").resize((s["width"], s["height"]))
    t0 = time.time()
    video = pipe(**args).frames[0]
    export_to_video(video, out + ".part.mp4", fps=FPS)
    os.replace(out + ".part.mp4", out)
    print(f"scene {s['n']}: done in {time.time() - t0:.0f} s")

In [ ]:
# 4) One file to download
shutil.make_archive("/kaggle/working/editpro-clips", "zip", OUT)
print("Finished:", len(os.listdir(OUT)), "clips. Download editpro-clips.zip from the Output panel (right side).")